 5년간에걸쳐약150개국가에서조사한행복지수와행복에관련이있을것으로예
상되는GDP나건강과같은지수를담고있다. 2015년부터2019년까지조사결과가
각각`2015.csv` ~ `2019.csv` 파일에 들어있다. 매해 조사 국가가 조금씩 차이가 있으
며컬럼가지수나이름또한조금씩다르다. 따라서데이터를분석하기위해컬럼이
름을직접출력해보고확인하여야한다

##Question
-  5년동안 행복지수가 지속적으로 상승하거나 적어도 전년도와 같은 나라를 찾아 출력 하시오

In [31]:
from pyspark import SparkContext
import urllib.request
import urllib.parse

In [32]:
for year in range(2015, 2020):
    path = f"data/World Hapiness Report/{year}.csv"
    encoded_path = urllib.parse.quote(path)

    url = f"https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/{encoded_path}"

    print(url)
    urllib.request.urlretrieve(url, f"{year}.csv")

https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2015.csv
https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2016.csv
https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2017.csv
https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2018.csv
https://raw.githubusercontent.com/nongaussian/class-2024-datamining/main/data/World%20Hapiness%20Report/2019.csv


In [33]:
sc = SparkContext.getOrCreate()

# only need (country, happiness_score)
for year in range(2015, 2020):
    rdd = sc.textFile(f"{year}.csv")
    print(year, rdd.first())

2015 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2016 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2017 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Generosity,Trust (Government Corruption)
2018 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)
2019 Country,Happiness Rank,Happiness Score,Economy (GDP per Capita),Family,Health (Life Expectancy),Freedom,Trust (Government Corruption)


In [34]:
# extract data

scores = {}

for year in range(2015, 2020):
    rdd = sc.textFile(f"{year}.csv")
    header = rdd.first()

    data = (
        rdd.filter(lambda row, h=header: row != h)
        .map(lambda row: row.split(","))
        .map(lambda x: (x[0], float(x[2])))
    )

    scores[year] = data

scores[2015].take(3)
scores[2019].take(3)

[('Finland', 7.769), ('Denmark', 7.6), ('Norway', 7.554)]

In [35]:
#join countries scores across years

joined = scores[2015].join(scores[2016]).join(scores[2017]).join(scores[2018]).join(scores[2019])

In [36]:
#clean up joined

flattened = joined.map(
    lambda x: (
        x[0],
        (
            x[1][0][0][0][0],
            x[1][0][0][0][1],
            x[1][0][0][1],
            x[1][0][1],
            x[1][1]
        )
    )
)

In [37]:
#행복지수가 지속적으로 상승하거나 적어도 전년도와 같은 나라

result = flattened.filter(
    lambda x:
        x[1][0] <= x[1][1]
        <= x[1][2]
        <= x[1][3]
        <= x[1][4]
)

In [38]:
result.collect()

[('Gabon', (3.896, 4.121, 4.465000153, 4.758, 4.799)),
 ('Malta', (6.302, 6.488, 6.52699995, 6.627, 6.726)),
 ('Senegal', (3.904, 4.219, 4.534999847, 4.631, 4.681)),
 ('Chad', (3.667, 3.763, 3.936000109, 4.301, 4.35)),
 ('Czech Republic', (6.505, 6.596, 6.609000206, 6.711, 6.852)),
 ('Cambodia', (3.819, 3.907, 4.168000221, 4.433, 4.7)),
 ('Tajikistan', (4.786, 4.996, 5.040999889, 5.199, 5.467)),
 ('Congo (Brazzaville)', (3.989, 4.236, 4.290999889, 4.559, 4.812)),
 ('Latvia', (5.098, 5.56, 5.849999905, 5.933, 5.94)),
 ('Finland', (7.406, 7.413, 7.468999863, 7.632, 7.769)),
 ('Dominican Republic', (4.885, 5.155, 5.230000019, 5.302, 5.425)),
 ('Honduras', (4.788, 4.871, 5.181000233, 5.504, 5.86)),
 ('Syria', (3.006, 3.069, 3.461999893, 3.462, 3.462)),
 ('Estonia', (5.429, 5.517, 5.611000061, 5.739, 5.893)),
 ('Mongolia', (4.874, 4.907, 4.954999924, 5.125, 5.285)),
 ('Poland', (5.791, 5.835, 5.97300005, 6.123, 6.182)),
 ('Togo', (2.839, 3.303, 3.494999886, 3.999, 4.085)),
 ('Slovakia', (5.

In [39]:
result.map(lambda x: x[0]).collect()

['Gabon',
 'Malta',
 'Senegal',
 'Chad',
 'Czech Republic',
 'Cambodia',
 'Tajikistan',
 'Congo (Brazzaville)',
 'Latvia',
 'Finland',
 'Dominican Republic',
 'Honduras',
 'Syria',
 'Estonia',
 'Mongolia',
 'Poland',
 'Togo',
 'Slovakia',
 'Hungary',
 'Cameroon',
 'Romania',
 'Benin',
 'Burkina Faso',
 'Serbia',
 'Ivory Coast',
 'Philippines',
 'Portugal',
 'Niger']